# Bank Loan Portfolio — Risk Analyst Report

Canonical Python companion to the Power BI dashboard and KPI contract.


## tl;dr

- Primary outcome metric: matured default rate among resolved loans.
- Review priority combines risk, funded exposure, and sample size.
- Results are descriptive monitoring signals, not approval rules or expected-loss estimates.


## Context & Methods

Metrics come from `src/risk_metrics.py` and `docs/metric_contract.md`. Only complete issue months are compared. Payment-date chronology is excluded until source semantics are verified.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

from IPython.display import display

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / 'data' / 'financial_loan.csv').exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError('Cannot find data/financial_loan.csv')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.risk_metrics import (
    complete_month_comparison, load_loan_data, portfolio_kpis, segment_summary
)
loans = load_loan_data(PROJECT_ROOT / 'data' / 'financial_loan.csv')


## Data


In [ ]:
print(f'Loan grain: {loans.id.nunique():,} unique IDs')
print(f'Issue-date range: {loans.issue_date.min():%Y-%m-%d} to {loans.issue_date.max():%Y-%m-%d}')
display(loans.head(10))


## Results

### 1. Portfolio health


In [ ]:
portfolio = portfolio_kpis(loans)
display(portfolio.rename('value').to_frame())


### 2. Grade risk and exposure


In [ ]:
grade_risk = segment_summary(loans, 'grade').sort_values('grade')
display(grade_risk[['grade','loan_count','funded_exposure','resolved_loans','matured_default_rate','ci_low','ci_high','risk_exposure_proxy','review_priority']])


### 3. State concentration


In [ ]:
state_risk = segment_summary(loans, 'address_state').sort_values('funded_exposure', ascending=False)
display(state_risk.head(10)[['address_state','funded_exposure','funded_exposure_share','resolved_loans','matured_default_rate','default_rate_vs_portfolio_pp','review_priority']])


### 4. Purpose × term review matrix


In [ ]:
purpose_term_risk = segment_summary(loans, ['purpose','term']).sort_values('risk_exposure_proxy', ascending=False)
display(purpose_term_risk.head(12)[['purpose','term','funded_exposure','resolved_loans','matured_default_rate','default_rate_vs_portfolio_pp','risk_exposure_proxy','review_priority']])


### 5. Latest complete issue-month comparison


In [ ]:
display(complete_month_comparison(loans))


## Takeaways

1. **Headline risk:** Prioritize a grade-controlled policy and pricing backtest for 60-month debt-consolidation loans; the segment combines material exposure, a 26.1% matured default rate, and higher rates than the 36-month comparison within Grades A–F.
2. **Opportunity hypothesis:** Investigate why Grade A has lower funded exposure than Grades B and C despite its lower observed default rate; do not treat this as an automatic expansion rule.
3. **State monitoring:** Treat Florida as the stronger risk hypothesis after accounting for grade mix, while treating California primarily as a concentration watch item.
4. **Supporting guardrail:** Retain DTI above 20% as a monitoring variable, not as evidence for a hard underwriting cutoff.
5. Keep payment timing, expected loss, profitability, and causal policy decisions out of scope until governed source data is available.
